# IE7275 Data Mining in Engineering - Group Project 1

**Project:** Emergency Room Triage Recommendation System (MIMIC-IV ED)

**Project Group 10**
- Sankalp Hegde
- Prarthana Prasanna Kumar
- Aarya Gadekar

This notebook includes full source code and embedded outputs/figures.


## 1. Metrics Snapshot (Embedded)
Source metric file at generation time: `reports/triage_metrics.json`


In [1]:
#

In [2]:
# Raw metrics JSON used for the table above
{
  "config": {
    "label_strategy": "acuity_le2",
    "threshold_metric": "f2",
    "threshold_range": [
      0.1,
      0.9,
      0.05
    ]
  },
  "baseline": {
    "k=5": {
      "recall@k": 0.7941218629796901,
      "precision@k": 0.8923655802366619
    },
    "k=10": {
      "recall@k": 0.9512565626692936,
      "precision@k": 0.8919360910927239
    },
    "roc_auc": 0.5232172808039138,
    "pr_auc": 0.5267722554370533,
    "f1": 0.13273998157013464,
    "accuracy": 0.5120372038911072
  },
  "context": {
    "best_params": {
      "model__C": 10.0,
      "model__penalty": "l2",
      "model__solver": "lbfgs"
    },
    "k=5": {
      "recall@k": 0.8237158044989151,
      "precision@k": 0.9228409798629853
    },
    "k=10": {
      "recall@k": 0.9604149897237053,
      "precision@k": 0.8993819425908902
    },
    "best_threshold": 0.15000000000000002,
    "best_threshold_score": 0.8497592075346986,
    "roc_auc": 0.8108510779046044,
    "pr_auc": 0.808688706566271,
    "f1": 0.7349219618311184,
    "f2": 0.8525079532293913,
    "accuracy": 0.6471257717878923
  },
  "graph": {
    "best_params": {},
    "k=5": {
      "recall@k": 0.7926974140018465,
      "precision@k": 0.8895699259566812
    },
    "k=10": {
      "recall@k": 0.9509140050381134,
      "precision@k": 0.8915208949125287
    },
    "best_threshold": 0.1,
    "best_threshold_score": 0.0,
    "roc_auc": 0.5,
    "pr_auc": 0.5125936183060342,
    "f1": 0.0,
    "f2": 0.0,
    "accuracy": 0.4874063816939658
  },
  "pairwise": {
    "k=5": {
      "recall@k": 0.7926974140018465,
      "precision@k": 0.8895699259566812
    },
    "k=10": {
      "recall@k": 0.9509140050381134,
      "precision@k": 0.8915208949125287
    },
    "best_threshold": 0.1,
    "best_threshold_score": 0.0,
    "roc_auc": 0.5,
    "pr_auc": 0.5125936183060342,
    "f1": 0.0,
    "f2": 0.0,
    "accuracy": 0.4874063816939658
  }
}


{'config': {'label_strategy': 'acuity_le2',
  'threshold_metric': 'f2',
  'threshold_range': [0.1, 0.9, 0.05]},
 'baseline': {'k=5': {'recall@k': 0.7941218629796901,
   'precision@k': 0.8923655802366619},
  'k=10': {'recall@k': 0.9512565626692936, 'precision@k': 0.8919360910927239},
  'roc_auc': 0.5232172808039138,
  'pr_auc': 0.5267722554370533,
  'f1': 0.13273998157013464,
  'accuracy': 0.5120372038911072},
 'context': {'best_params': {'model__C': 10.0,
   'model__penalty': 'l2',
   'model__solver': 'lbfgs'},
  'k=5': {'recall@k': 0.8237158044989151, 'precision@k': 0.9228409798629853},
  'k=10': {'recall@k': 0.9604149897237053, 'precision@k': 0.8993819425908902},
  'best_threshold': 0.15000000000000002,
  'best_threshold_score': 0.8497592075346986,
  'roc_auc': 0.8108510779046044,
  'pr_auc': 0.808688706566271,
  'f1': 0.7349219618311184,
  'f2': 0.8525079532293913,
  'accuracy': 0.6471257717878923},
 'graph': {'best_params': {},
  'k=5': {'recall@k': 0.7926974140018465, 'precision@k

## Full Source Code: Data Preprocessing
File: `src/data_preprocessing.py`


In [3]:
import os
import pandas as pd
import numpy as np

def _pick_existing_path(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

def load_all_files():
    """
    Loads all 6 MIMIC-IV ED files. Supports both full and demo filenames.
    """
    # Fixed filenames: 'edstays' (no underscore) and 'vitalsign' (no 's')
    paths = {
        'triage': _pick_existing_path([
            'data/raw/triage.csv',
            'data/raw/triage.csv'
        ]),
        'vitals': _pick_existing_path([
            'data/raw/vitalsign.csv',
            'data/raw/vitalsign.csv'
        ]),
        'edstays': _pick_existing_path([
            'data/raw/edstays.csv',
            'data/raw/edstays.csv'
        ]),
        'diagnosis': _pick_existing_path([
            'data/raw/diagnosis.csv',
            'data/raw/diagnosis.csv'
        ]),
        'meds': _pick_existing_path([
            'data/raw/medrecon.csv',
            'data/raw/medrecon.csv'
        ]),
        'pyxis': _pick_existing_path([
            'data/raw/pyxis.csv',
            'data/raw/pyxis.csv'
        ])
    }
    
    dfs = {}
    for name, path in paths.items():
        try:
            if path is None:
                raise FileNotFoundError
            dfs[name] = pd.read_csv(path)
            print(f"Loaded {name}: {dfs[name].shape} from {path}")
        except FileNotFoundError:
            print(f"Error: Missing {name} file. Expected in data/raw/")
            
    return dfs

def preprocess_master(dfs):
    """
    Merges all files and creates new features for Data Mining IE 7275.
    """
    # 1. Start with Triage and Vitals
    triage = dfs['triage']
    vitals = dfs['vitals']
    
    # FIX: Use 'acuity' (MIMIC column) instead of 'triage_level'
    if 'acuity' in triage.columns:
        triage['acuity'] = triage['acuity'].fillna(triage['acuity'].mode()[0])
    
    # 2. Clean Vitals
    numeric_vitals = vitals.select_dtypes(include=[np.number])
    vitals_cleaned = vitals.fillna(numeric_vitals.median())
    
    # 3. Master Merge (MIMIC-IV uses subject_id and stay_id)
    master = triage.merge(vitals_cleaned, on=['subject_id', 'stay_id'], how='left')

    # 4. Feature: Diagnosis Count (From diagnosis_demo.csv)
    if 'diagnosis' in dfs:
        diag_counts = dfs['diagnosis'].groupby('subject_id').size().reset_index(name='num_prior_cond')
        master = master.merge(diag_counts, on='subject_id', how='left')

    # 5. Feature: Home Med Count (From medrecon_demo.csv)
    if 'meds' in dfs:
        med_counts = dfs['meds'].groupby('subject_id').size().reset_index(name='num_home_meds')
        master = master.merge(med_counts, on='subject_id', how='left')

    # 6. Feature: ER Meds Administered (From pyxis_demo.csv)
    if 'pyxis' in dfs:
        pyxis_counts = dfs['pyxis'].groupby('stay_id').size().reset_index(name='er_meds_given')
        master = master.merge(pyxis_counts, on='stay_id', how='left')

    # 7. Add ED Stay details (From edstays_demo.csv)
    if 'edstays' in dfs:
        master = master.merge(dfs['edstays'], on=['subject_id', 'stay_id'], how='left')

    # Final cleanup: Replace NaN counts with 0
    fill_cols = ['num_prior_cond', 'num_home_meds', 'er_meds_given']
    for col in fill_cols:
        if col in master.columns:
            master[col] = master[col].fillna(0)
    
    return master

if __name__ == "__main__":
    # Step 1: Load everything
    all_data = load_all_files()
    
    if all_data:
        # Step 2: Process and Merge
        final_df = preprocess_master(all_data)
        
        # Step 3: Save the Master Dataset
        output_file = 'data/processed/master_dataset.csv'
        final_df.to_csv(output_file, index=False)
        
        print("\n--- EMERGEN AI: PREPROCESSING COMPLETE ---")
        print(f"Master file created at: {output_file}")
        print(f"Final dataset shape: {final_df.shape}")
        print("\nFirst 5 rows of merged data:")
        print(final_df.head())


Error: Missing triage file. Expected in data/raw/
Error: Missing vitals file. Expected in data/raw/
Error: Missing edstays file. Expected in data/raw/
Error: Missing diagnosis file. Expected in data/raw/
Error: Missing meds file. Expected in data/raw/
Error: Missing pyxis file. Expected in data/raw/


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Full Source Code: Feature Engineering
File: `src/feature_engineering.py`


In [4]:
import os
import pandas as pd
import numpy as np

def calculate_patient_features(df):
    """Calculates clinical indices from vitals using the merged column names"""
    # Using the '_y' suffix names seen in your master_dataset spreadsheet
    # Shock Index: HR / SBP
    df['shock_index'] = df['heartrate_y'] / df['sbp_y']
    
    # Pulse Pressure: SBP - DBP
    df['pulse_pressure'] = df['sbp_y'] - df['dbp_y']
    
    # Complexity Score: Number of conditions + home meds
    # These names likely stayed the same, but we use fillna to be safe
    df['complexity_score'] = df['num_prior_cond'].fillna(0) + df['num_home_meds'].fillna(0)
    
    return df

def calculate_context_features(df):
    """Calculates ER congestion/resources as per your flowchart"""
    # Using 'charttime' from your spreadsheet to calculate arrival patterns
    if 'charttime' in df.columns:
        df['charttime'] = pd.to_datetime(df['charttime'])
        df['arrival_hour'] = df['charttime'].dt.hour
        df['arrival_day'] = df['charttime'].dt.dayofweek
    return df

if __name__ == "__main__":
    # Load the master dataset
    df = pd.read_csv('data/processed/master_dataset.csv')
    
    # Run the pipeline
    df = calculate_patient_features(df)
    df = calculate_context_features(df)
    
    # Cleanup: Replace infinities from division and fill remaining NaNs
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    # Drop datetime columns to speed up CSV writing (features already extracted)
    datetime_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns
    if len(datetime_cols) > 0:
        df = df.drop(columns=list(datetime_cols))

    # Save the 'Model Ready' file
    output_path = os.getenv("MODEL_READY_PATH", "data/processed/model_ready.csv")
    output_format = os.getenv("MODEL_READY_FORMAT", "csv").lower()
    if output_format == "parquet":
        try:
            df.to_parquet(output_path, index=False)
            print(f"✅ Feature Engineering Complete. '{output_path}' is ready for training!")
        except Exception:
            output_format = "csv"

    if output_format == "csv":
        df.to_csv(
            output_path,
            index=False,
            date_format="%Y-%m-%d %H:%M:%S",
            chunksize=200_000,
        )
        print(f"✅ Feature Engineering Complete. '{output_path}' is ready for training!")
    print(f"Engineered columns: ['shock_index', 'pulse_pressure', 'arrival_hour']")


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/master_dataset.csv'

## Full Source Code: EDA Visualization
File: `src/eda_visualization.py`


In [ ]:
import os
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns


DATA_PATH = "data/processed/master_dataset.csv"
OUT_DIR = "reports/eda"
MAX_ROWS = int(os.getenv("EDA_MAX_ROWS", "200000"))
PAIRPLOT_MAX_ROWS = int(os.getenv("EDA_PAIRPLOT_MAX_ROWS", "3000"))
ENABLE_PAIRPLOT = os.getenv("EDA_PAIRPLOT", "0") == "1"


def unify_vitals(df: pd.DataFrame) -> pd.DataFrame:
    pairs = [
        ("temperature", "temperature_y", "temperature_x"),
        ("heartrate", "heartrate_y", "heartrate_x"),
        ("resprate", "resprate_y", "resprate_x"),
        ("o2sat", "o2sat_y", "o2sat_x"),
        ("sbp", "sbp_y", "sbp_x"),
        ("dbp", "dbp_y", "dbp_x"),
        ("pain", "pain_y", "pain_x"),
    ]
    for base, y_col, x_col in pairs:
        if y_col in df.columns and x_col in df.columns:
            df[base] = df[y_col].combine_first(df[x_col])
        elif y_col in df.columns:
            df[base] = df[y_col]
        elif x_col in df.columns:
            df[base] = df[x_col]
    return df


def add_time_features(df: pd.DataFrame, time_col: str = "intime") -> pd.DataFrame:
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df["arrival_hour"] = df[time_col].dt.hour
    df["arrival_day"] = df[time_col].dt.dayofweek
    return df


def save_plot(fig, filename: str, pdf: PdfPages = None):
    path = os.path.join(OUT_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    if pdf is not None:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def plot_distribution(df: pd.DataFrame, col: str, title: str, filename: str, pdf: PdfPages = None):
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df[col].dropna(), kde=True, bins=50, ax=ax)
    ax.set_title(title)
    ax.set_xlabel(col)
    save_plot(fig, filename, pdf)


def plot_count(df: pd.DataFrame, col: str, title: str, filename: str, top_n: int = 10, pdf: PdfPages = None):
    fig, ax = plt.subplots(figsize=(7, 4))
    counts = df[col].fillna("UNKNOWN").value_counts().head(top_n)
    sns.barplot(x=counts.values, y=counts.index, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Count")
    save_plot(fig, filename, pdf)


def plot_missingness(df: pd.DataFrame, filename: str, pdf: PdfPages = None):
    miss = df.isna().mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(x=miss.values[:20], y=miss.index[:20], ax=ax)
    ax.set_title("Top 20 Missingness Rates")
    ax.set_xlabel("Missing Fraction")
    save_plot(fig, filename, pdf)


def plot_correlation(df: pd.DataFrame, cols: List[str], filename: str, pdf: PdfPages = None):
    corr = df[cols].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=False, cmap="coolwarm", ax=ax)
    ax.set_title("Correlation Heatmap (Vitals + Features)")
    save_plot(fig, filename, pdf)


def plot_boxplots(df: pd.DataFrame, cols: List[str], filename: str, pdf: PdfPages = None):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=df[cols], orient="h", ax=ax)
    ax.set_title("Vital Sign Boxplots (Outliers)")
    save_plot(fig, filename, pdf)


def plot_violin_by_acuity(df: pd.DataFrame, col: str, filename: str, pdf: PdfPages = None):
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.violinplot(x="acuity", y=col, data=df, ax=ax)
    ax.set_title(f"{col} by Acuity")
    save_plot(fig, filename, pdf)


def plot_pairplot(df: pd.DataFrame, cols: List[str], filename: str):
    pp = sns.pairplot(df[cols].dropna())
    pp.fig.suptitle("Pairplot of Key Vitals", y=1.02)
    path = os.path.join(OUT_DIR, filename)
    pp.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(pp.fig)


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    df = pd.read_csv(DATA_PATH)
    df = unify_vitals(df)
    df = add_time_features(df, "intime")

    if len(df) > MAX_ROWS:
        df = df.sample(n=MAX_ROWS, random_state=42)

    pdf_path = os.path.join(OUT_DIR, "eda_report.pdf")
    with PdfPages(pdf_path) as pdf:
        # Basic distributions
        plot_distribution(df, "temperature", "Temperature Distribution", "dist_temperature.png", pdf)
        plot_distribution(df, "heartrate", "Heart Rate Distribution", "dist_heartrate.png", pdf)
        plot_distribution(df, "resprate", "Respiratory Rate Distribution", "dist_resprate.png", pdf)
        plot_distribution(df, "o2sat", "O2 Saturation Distribution", "dist_o2sat.png", pdf)
        plot_distribution(df, "sbp", "Systolic BP Distribution", "dist_sbp.png", pdf)
        plot_distribution(df, "dbp", "Diastolic BP Distribution", "dist_dbp.png", pdf)

        # Categorical counts
        plot_count(df, "disposition", "Disposition Counts", "count_disposition.png", pdf=pdf)
        plot_count(df, "acuity", "Acuity Counts", "count_acuity.png", pdf=pdf)
        plot_count(df, "arrival_transport", "Arrival Transport", "count_transport.png", pdf=pdf)
        plot_count(df, "chiefcomplaint", "Top Chief Complaints", "count_chiefcomplaint.png", top_n=15, pdf=pdf)

        # Time patterns
        plot_count(df, "arrival_hour", "Arrivals by Hour", "count_arrival_hour.png", top_n=24, pdf=pdf)
        plot_count(df, "arrival_day", "Arrivals by Day of Week (0=Mon)", "count_arrival_day.png", top_n=7, pdf=pdf)

        # Missingness + correlation
        plot_missingness(df, "missingness_top20.png", pdf)
        corr_cols = [
            "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp",
            "num_prior_cond", "num_home_meds", "er_meds_given"
        ]
        corr_cols = [c for c in corr_cols if c in df.columns]
        plot_correlation(df, corr_cols, "corr_heatmap.png", pdf)

        # Outlier boxplots
        vitals_cols = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]
        vitals_cols = [c for c in vitals_cols if c in df.columns]
        plot_boxplots(df, vitals_cols, "boxplot_vitals.png", pdf)

        # Violin plots by acuity
        for col in ["heartrate", "resprate", "o2sat", "sbp"]:
            if col in df.columns:
                plot_violin_by_acuity(df, col, f"violin_{col}_by_acuity.png", pdf)

    # Pairplot is expensive on large datasets; enable explicitly with EDA_PAIRPLOT=1
    if ENABLE_PAIRPLOT:
        pair_cols = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]
        pair_cols = [c for c in pair_cols if c in df.columns]
        if len(pair_cols) >= 2:
            pp_df = df[pair_cols].dropna()
            if len(pp_df) > PAIRPLOT_MAX_ROWS:
                pp_df = pp_df.sample(n=PAIRPLOT_MAX_ROWS, random_state=42)
            plot_pairplot(pp_df, pair_cols, "pairplot_vitals.png")

    # Summary table
    summary = df.describe(include="all").transpose()
    summary.to_csv(os.path.join(OUT_DIR, "summary_stats.csv"))

    print(f"EDA plots saved to {OUT_DIR}")
    print(f"PDF report saved to {pdf_path}")


if __name__ == "__main__":
    main()


## Full Source Code: Modeling and Evaluation Pipeline
File: `src/triage_pipeline.py`


In [ ]:
import json
import os
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, f1_score, accuracy_score
import matplotlib.pyplot as plt


@dataclass
class Config:
    data_path: str = "data/processed/master_dataset.csv"
    output_dir: str = "reports"
    time_col: str = "intime"
    test_size: float = 0.2
    random_state: int = 42
    k_values: Tuple[int, ...] = (5, 10)
    decision_threshold: float = 0.4
    label_strategy: str = "acuity_le2"
    threshold_metric: str = "f2"
    threshold_min: float = 0.1
    threshold_max: float = 0.9
    threshold_step: float = 0.05
    mf_factors: int = 16


def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df


def unify_vitals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prefer *_y columns (from vitals) and fallback to *_x (from triage).
    Produces unified columns without suffixes.
    """
    pairs = [
        ("temperature", "temperature_y", "temperature_x"),
        ("heartrate", "heartrate_y", "heartrate_x"),
        ("resprate", "resprate_y", "resprate_x"),
        ("o2sat", "o2sat_y", "o2sat_x"),
        ("sbp", "sbp_y", "sbp_x"),
        ("dbp", "dbp_y", "dbp_x"),
        ("pain", "pain_y", "pain_x"),
    ]
    for base, y_col, x_col in pairs:
        if y_col in df.columns and x_col in df.columns:
            df[base] = df[y_col].combine_first(df[x_col])
        elif y_col in df.columns:
            df[base] = df[y_col]
        elif x_col in df.columns:
            df[base] = df[x_col]
    return df


def add_time_features(df: pd.DataFrame, time_col: str) -> pd.DataFrame:
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df["arrival_hour"] = df[time_col].dt.hour
    df["arrival_day"] = df[time_col].dt.dayofweek
    return df


def add_patient_features(df: pd.DataFrame) -> pd.DataFrame:
    df["shock_index"] = df["heartrate"] / df["sbp"]
    df["pulse_pressure"] = df["sbp"] - df["dbp"]
    df["complexity_score"] = df["num_prior_cond"].fillna(0) + df["num_home_meds"].fillna(0)
    return df


def simulate_context(df: pd.DataFrame, time_col: str) -> pd.DataFrame:
    """
    More realistic congestion simulation:
    - arrivals_per_hour: count per hour
    - rolling_arrivals_6h: rolling 6-hour arrivals
    - active_patients: number of ongoing ED stays at arrival time
    - bed_capacity: 85th percentile of active patients
    - congestion_score: active_patients / bed_capacity (capped)
    - resource_availability: inverse of congestion
    - wait_time_proxy: backlog * 5 minutes
    """
    df = df.sort_values(time_col).copy()
    df["arrival_hour_bucket"] = df[time_col].dt.floor("h")
    arrivals = df.groupby("arrival_hour_bucket").size().rename("arrivals_per_hour")
    df = df.merge(arrivals, on="arrival_hour_bucket", how="left")

    rolling = arrivals.rolling(window=6, min_periods=1).sum().rename("rolling_arrivals_6h")
    df = df.merge(rolling, on="arrival_hour_bucket", how="left")

    # Active patient count at arrival
    times = df[time_col].values
    outtimes = pd.to_datetime(df["outtime"], errors="coerce").values
    events = []
    for t_in, t_out in zip(times, outtimes):
        if pd.isna(t_in) or pd.isna(t_out):
            continue
        events.append((t_in, 1))
        events.append((t_out, -1))
    events.sort(key=lambda x: x[0])

    active_counts = np.zeros(len(df), dtype=int)
    order = np.argsort(times)
    running = 0
    ei = 0
    for idx in order:
        t = times[idx]
        while ei < len(events) and events[ei][0] < t:
            running += events[ei][1]
            ei += 1
        active_counts[idx] = max(running, 0)

    df["active_patients"] = active_counts
    bed_capacity = np.percentile(active_counts, 85) if len(active_counts) else 1.0
    bed_capacity = max(bed_capacity, 1.0)
    df["bed_capacity"] = bed_capacity
    df["congestion_score"] = np.minimum(df["active_patients"] / bed_capacity, 1.5)
    df["resource_availability"] = np.maximum(0.0, 1.0 - df["congestion_score"])
    df["wait_time_proxy"] = np.maximum(0, df["active_patients"] - bed_capacity) * 5
    return df


def create_label(df: pd.DataFrame, strategy: str) -> pd.DataFrame:
    """
    Label strategies:
    - acuity_le2: critical if acuity <= 2
    - admit_or_transfer: critical if disposition in {ADMITTED, TRANSFER}
    - combined: acuity<=2 OR admit/transfer
    """
    df = df.copy()
    if strategy == "acuity_le2" and "acuity" in df.columns:
        df["is_critical"] = (df["acuity"].fillna(99) <= 2).astype(int)
    elif strategy == "combined" and "acuity" in df.columns:
        critical_dispo = {"ADMITTED", "TRANSFER"}
        df["is_critical"] = (
            (df["acuity"].fillna(99) <= 2) | (df["disposition"].isin(critical_dispo))
        ).astype(int)
    else:
        critical_dispo = {"ADMITTED", "TRANSFER"}
        df["is_critical"] = df["disposition"].isin(critical_dispo).astype(int)
    return df


def time_split_three(
    df: pd.DataFrame, time_col: str, train_frac: float = 0.6, val_frac: float = 0.2
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = df.sort_values(time_col).copy()
    n = len(df)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))
    train = df.iloc[:train_end]
    val = df.iloc[train_end:val_end]
    test = df.iloc[val_end:]
    return train, val, test


def rule_based_score(row: pd.Series) -> int:
    """
    Rule-based triage score using vital sign thresholds.
    Higher score = higher urgency.
    """
    score = 0
    if row["sbp"] < 90:
        score += 2
    if row["o2sat"] < 92:
        score += 2
    if row["heartrate"] > 130:
        score += 1
    if row["resprate"] > 30:
        score += 1
    if row["temperature"] > 103 or row["temperature"] < 95:
        score += 1
    return score


def evaluate_at_k(df: pd.DataFrame, score_col: str, label_col: str, group_col: str, k: int) -> Dict[str, float]:
    """
    Compute Recall@k and Precision@k by grouping (e.g., hour buckets).
    """
    recalls = []
    precisions = []
    for _, group in df.groupby(group_col):
        if group[label_col].sum() == 0:
            continue
        ranked = group.sort_values(score_col, ascending=False)
        top_k = ranked.head(k)
        recall = top_k[label_col].sum() / group[label_col].sum()
        precision = top_k[label_col].sum() / min(k, len(group))
        recalls.append(recall)
        precisions.append(precision)
    if not recalls:
        return {"recall@k": 0.0, "precision@k": 0.0}
    return {
        "recall@k": float(np.mean(recalls)),
        "precision@k": float(np.mean(precisions)),
    }


def evaluate_ndcg_at_k(df: pd.DataFrame, score_col: str, label_col: str, group_col: str, k: int) -> Dict[str, float]:
    """
    Compute NDCG@k on grouped rankings with binary relevance labels.
    """
    ndcgs = []
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    for _, group in df.groupby(group_col):
        labels = group[label_col].values
        if labels.sum() == 0:
            continue
        ranked = group.sort_values(score_col, ascending=False)
        rel = ranked[label_col].values[:k].astype(float)
        dcg = np.sum((2 ** rel - 1) * discounts[: len(rel)])
        ideal_rel = np.sort(labels)[::-1][:k].astype(float)
        idcg = np.sum((2 ** ideal_rel - 1) * discounts[: len(ideal_rel)])
        if idcg > 0:
            ndcgs.append(dcg / idcg)
    if not ndcgs:
        return {"ndcg@k": 0.0}
    return {"ndcg@k": float(np.mean(ndcgs))}


def fbeta_score_safe(y_true: np.ndarray, y_pred: np.ndarray, beta: float = 2.0) -> float:
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    if tp == 0:
        return 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    beta2 = beta ** 2
    if precision == 0 and recall == 0:
        return 0.0
    return (1 + beta2) * (precision * recall) / (beta2 * precision + recall)


def find_best_threshold(
    scores: np.ndarray,
    y_true: np.ndarray,
    metric: str,
    t_min: float,
    t_max: float,
    t_step: float,
) -> Tuple[float, float]:
    best_t = t_min
    best_score = -1.0
    thresholds = np.arange(t_min, t_max + 1e-9, t_step)
    for t in thresholds:
        y_pred = (scores > t).astype(int)
        if metric == "f2":
            score = fbeta_score_safe(y_true, y_pred, beta=2.0)
        elif metric == "f1":
            score = f1_score(y_true, y_pred)
        else:
            score = fbeta_score_safe(y_true, y_pred, beta=2.0)
        if score > best_score:
            best_score = score
            best_t = t
    return best_t, best_score


def build_pairwise_dataset(
    df: pd.DataFrame,
    feature_cols: List[str],
    group_col: str,
    max_pairs_per_group: int = 200,
    random_state: int = 42,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(random_state)
    X_list = []
    y_list = []
    for _, group in df.groupby(group_col):
        pos = group[group["is_critical"] == 1]
        neg = group[group["is_critical"] == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        pairs = []
        for _, p in pos.iterrows():
            for _, n in neg.iterrows():
                pairs.append((p, n))
        if len(pairs) > max_pairs_per_group:
            idx = rng.choice(len(pairs), size=max_pairs_per_group, replace=False)
            pairs = [pairs[i] for i in idx]
        for p, n in pairs:
            X_list.append(p[feature_cols].values - n[feature_cols].values)
            y_list.append(1)
            X_list.append(n[feature_cols].values - p[feature_cols].values)
            y_list.append(0)
    if not X_list:
        return np.empty((0, len(feature_cols))), np.empty((0,))
    return np.vstack(X_list), np.array(y_list)


def train_pairwise_ranker(
    train: pd.DataFrame,
    val: pd.DataFrame,
    feature_cols: List[str],
    group_col: str,
) -> Tuple[StandardScaler, LogisticRegression | None]:
    X_train, y_train = build_pairwise_dataset(train, feature_cols, group_col)
    scaler = StandardScaler()
    if X_train.shape[0] == 0:
        scaler.fit(train[feature_cols].values)
        return scaler, None
    X_train = scaler.fit_transform(X_train)
    model = LogisticRegression(max_iter=2000, class_weight="balanced", fit_intercept=False)
    model.fit(X_train, y_train)
    return scaler, model


def score_pairwise(
    df: pd.DataFrame,
    feature_cols: List[str],
    scaler: StandardScaler,
    model: LogisticRegression | None,
) -> np.ndarray:
    X = df[feature_cols].values
    X = scaler.transform(X)
    if model is None:
        return np.zeros(len(df))
    return model.decision_function(X)


def train_matrix_factorization_model(train: pd.DataFrame, n_factors: int = 16) -> Dict[str, object]:
    """
    Matrix-factorization style recommender over (subject_id, chiefcomplaint)
    interactions with implicit criticality signal.
    """
    mf_df = train[["subject_id", "chiefcomplaint", "is_critical"]].copy()
    mf_df["chiefcomplaint"] = mf_df["chiefcomplaint"].astype("string").fillna("UNKNOWN").astype(str)
    user_vals = mf_df["subject_id"].astype(str).values
    item_vals = mf_df["chiefcomplaint"].values
    user_codes, user_uniques = pd.factorize(user_vals, sort=True)
    item_codes, item_uniques = pd.factorize(item_vals, sort=True)
    data = mf_df["is_critical"].astype(float).values
    mat = sparse.csr_matrix((data, (user_codes, item_codes)), shape=(len(user_uniques), len(item_uniques)))
    if min(mat.shape) <= 2:
        return {
            "user_to_idx": {u: i for i, u in enumerate(user_uniques)},
            "item_to_idx": {i: j for j, i in enumerate(item_uniques)},
            "user_factors": np.zeros((len(user_uniques), 1)),
            "item_factors": np.zeros((len(item_uniques), 1)),
            "user_bias": np.zeros(len(user_uniques)),
            "item_bias": np.zeros(len(item_uniques)),
            "global_mean": float(np.mean(data) if len(data) else 0.5),
            "n_factors": 1,
        }
    k = max(2, min(n_factors, min(mat.shape) - 1))
    svd = TruncatedSVD(n_components=k, random_state=42)
    user_factors = svd.fit_transform(mat)
    item_factors = svd.components_.T
    global_mean = float(np.mean(data) if len(data) else 0.5)
    user_bias = np.asarray(mat.mean(axis=1)).ravel() - global_mean
    item_bias = np.asarray(mat.mean(axis=0)).ravel() - global_mean
    return {
        "user_to_idx": {u: i for i, u in enumerate(user_uniques)},
        "item_to_idx": {i: j for j, i in enumerate(item_uniques)},
        "user_factors": user_factors,
        "item_factors": item_factors,
        "user_bias": user_bias,
        "item_bias": item_bias,
        "global_mean": global_mean,
        "n_factors": k,
    }


def score_matrix_factorization(df: pd.DataFrame, mf_model: Dict[str, object]) -> np.ndarray:
    score = np.full(len(df), mf_model["global_mean"], dtype=float)
    users = df["subject_id"].astype(str).map(mf_model["user_to_idx"])
    items = df["chiefcomplaint"].astype("string").fillna("UNKNOWN").astype(str).map(mf_model["item_to_idx"])
    user_known = users.notna().values
    item_known = items.notna().values
    both_known = user_known & item_known
    if user_known.any():
        u_idx = users[user_known].astype(int).values
        score[user_known] += mf_model["user_bias"][u_idx]
    if item_known.any():
        i_idx = items[item_known].astype(int).values
        score[item_known] += mf_model["item_bias"][i_idx]
    if both_known.any():
        b_idx = np.where(both_known)[0]
        u_idx = users.iloc[b_idx].astype(int).values
        i_idx = items.iloc[b_idx].astype(int).values
        score[b_idx] += np.sum(
            mf_model["user_factors"][u_idx] * mf_model["item_factors"][i_idx],
            axis=1,
        )
    # Convert to [0,1] probability-like score for threshold search.
    score = 1.0 / (1.0 + np.exp(-np.clip(score, -20, 20)))
    return score


def train_context_model(
    train: pd.DataFrame,
    test: pd.DataFrame,
    feature_cols: List[str],
    cat_cols: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, Pipeline, Dict[str, float], Dict[str, np.ndarray]]:
    X_train = train[feature_cols + cat_cols].copy()
    y_train = train["is_critical"]
    X_test = test[feature_cols + cat_cols].copy()

    # Ensure categorical columns are uniformly string-typed for OneHotEncoder
    for col in cat_cols:
        if col in X_train.columns:
            X_train[col] = X_train[col].astype("string").fillna("UNKNOWN")
        if col in X_test.columns:
            X_test[col] = X_test[col].astype("string").fillna("UNKNOWN")

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", Pipeline(steps=[("scaler", StandardScaler())]), feature_cols),
        ]
    )

    base_model = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf = Pipeline(steps=[("pre", preprocessor), ("model", base_model)])

    param_grid = {
        "model__C": [0.1, 1.0, 10.0],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"],
    }
    # Optional: sample for faster grid search on very large datasets
    sample_n = int(os.getenv("CONTEXT_MAX_ROWS", "0"))
    if sample_n > 0 and len(X_train) > sample_n:
        sample_idx = X_train.sample(n=sample_n, random_state=42).index
        X_grid = X_train.loc[sample_idx]
        y_grid = y_train.loc[sample_idx]
    else:
        X_grid = X_train
        y_grid = y_train

    grid = GridSearchCV(
        clf,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=3,
        n_jobs=1,
    )
    grid.fit(X_grid, y_grid)
    # Refit best params on full training data
    best_params = grid.best_params_
    clf = Pipeline(steps=[("pre", preprocessor), ("model", base_model)])
    clf.set_params(**best_params)
    clf.fit(X_train, y_train)
    test = test.copy()
    test["context_score"] = clf.predict_proba(X_test)[:, 1]
    train = train.copy()
    train["context_score"] = clf.predict_proba(X_train)[:, 1]
    return train, test, clf, best_params, grid.cv_results_


def build_graph_embeddings(df: pd.DataFrame, dim: int = 16) -> pd.DataFrame:
    """
    Simple graph embedding using SVD on patient-attribute bipartite matrix.
    Nodes: patients; attributes: chief complaint, arrival_transport, race.
    """
    attrs = ["chiefcomplaint", "arrival_transport", "race"]
    df = df.copy()
    df["patient_id"] = df["stay_id"].astype(str)

    for col in attrs:
        df[col] = df[col].fillna("UNKNOWN")

    attr_values = []
    for col in attrs:
        vals = df[col].astype(str).unique().tolist()
        attr_values.extend([f"{col}:{v}" for v in vals])
    attr_index = {v: i for i, v in enumerate(sorted(attr_values))}

    patient_index = {pid: i for i, pid in enumerate(df["patient_id"].unique())}
    mat = np.zeros((len(patient_index), len(attr_index)), dtype=np.float32)

    for _, row in df.iterrows():
        p = patient_index[row["patient_id"]]
        for col in attrs:
            key = f"{col}:{row[col]}"
            mat[p, attr_index[key]] = 1.0

    u, s, _ = np.linalg.svd(mat, full_matrices=False)
    emb = u[:, :dim] * s[:dim]

    emb_df = pd.DataFrame(emb, columns=[f"emb_{i}" for i in range(dim)])
    emb_df["patient_id"] = list(patient_index.keys())
    return emb_df


def train_graph_model(
    train: pd.DataFrame,
    test: pd.DataFrame,
    emb_df: pd.DataFrame,
    base_cols: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, Tuple[StandardScaler, LogisticRegression] | None, Dict[str, float], Dict[str, np.ndarray]]:
    train = train.copy()
    test = test.copy()
    train["patient_id"] = train["stay_id"].astype(str)
    test["patient_id"] = test["stay_id"].astype(str)

    train = train.merge(emb_df, on="patient_id", how="left")
    test = test.merge(emb_df, on="patient_id", how="left")

    emb_cols = [c for c in train.columns if c.startswith("emb_")]
    feature_cols = base_cols + emb_cols

    X_train = train[feature_cols].fillna(0)
    y_train = train["is_critical"]
    X_test = test[feature_cols].fillna(0)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    base_model = LogisticRegression(max_iter=2000, class_weight="balanced")
    param_grid = {
        "C": [0.1, 1.0, 10.0],
        "penalty": ["l2"],
        "solver": ["lbfgs"],
    }
    grid = GridSearchCV(
        base_model,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=3,
        n_jobs=1,
    )
    grid.fit(X_train_scaled, y_train)
    model = grid.best_estimator_
    train["graph_score"] = model.predict_proba(X_train_scaled)[:, 1]
    test["graph_score"] = model.predict_proba(X_test_scaled)[:, 1]
    return train, test, (scaler, model), grid.best_params_, grid.cv_results_


def plot_model_selection(cv_results: Dict[str, np.ndarray], title: str, out_path: str):
    params = cv_results.get("params", [])
    mean_scores = cv_results.get("mean_test_score", [])
    if not params or len(mean_scores) == 0:
        return

    c_vals = [p.get("model__C", p.get("C")) for p in params]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(c_vals, mean_scores, marker="o")
    ax.set_xscale("log")
    ax.set_xlabel("C (log scale)")
    ax.set_ylabel("Mean CV ROC-AUC")
    ax.set_title(title)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def main():
    cfg = Config()
    fast_mode = os.getenv("FAST_MODE", "0") == "1"
    max_rows = int(os.getenv("PIPELINE_MAX_ROWS", "0"))

    # Step 1: Load data
    df = load_data(cfg.data_path)
    if max_rows > 0 and len(df) > max_rows:
        # Keep temporal realism while capping runtime for experimentation runs.
        df = df.sort_values(cfg.time_col).tail(max_rows).copy()

    # Step 2: Unify vitals + time features
    df = unify_vitals(df)
    df = add_time_features(df, cfg.time_col)

    # Step 3: Feature engineering
    df = add_patient_features(df)
    df = simulate_context(df, cfg.time_col)

    # Step 4: Label creation
    df = create_label(df, cfg.label_strategy)

    # Step 5: Basic cleanup
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    # Step 6: Train/val/test split
    train, val, test = time_split_three(df, cfg.time_col)

    # Step 7: Baseline rule-based triage
    test = test.copy()
    val = val.copy()
    test["rule_score"] = test.apply(rule_based_score, axis=1)
    val["rule_score"] = val.apply(rule_based_score, axis=1)

    # Step 8: Context-aware model
    numeric_features = [
        "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp",
        "shock_index", "pulse_pressure", "complexity_score",
        "arrivals_per_hour", "rolling_arrivals_6h", "active_patients",
        "bed_capacity", "congestion_score", "resource_availability", "wait_time_proxy",
        "arrival_hour", "arrival_day"
    ]
    categorical_features = ["gender", "race", "arrival_transport", "chiefcomplaint"]
    for col in categorical_features:
        if col in train.columns:
            train[col] = train[col].astype("string").fillna("UNKNOWN")
        if col in val.columns:
            val[col] = val[col].astype("string").fillna("UNKNOWN")
        if col in test.columns:
            test[col] = test[col].astype("string").fillna("UNKNOWN")
    train, test, context_model, context_params, context_cv = train_context_model(
        train, test, numeric_features, categorical_features
    )
    # Score validation set for threshold tuning
    X_val = val[numeric_features + categorical_features].copy()
    val["context_score"] = context_model.predict_proba(X_val)[:, 1]

    # Step 8b: Matrix factorization recommender
    mf_model = train_matrix_factorization_model(train, cfg.mf_factors)
    val["mf_score"] = score_matrix_factorization(val, mf_model)
    test["mf_score"] = score_matrix_factorization(test, mf_model)

    # Step 9: Graph-based model
    if not fast_mode:
        emb_df = build_graph_embeddings(df)
        base_graph_cols = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "congestion_score"]
        train, test, graph_model, graph_params, graph_cv = train_graph_model(
            train, test, emb_df, base_graph_cols
        )
        val = val.copy()
        val["patient_id"] = val["stay_id"].astype(str)
        val = val.merge(emb_df, on="patient_id", how="left")
        emb_cols = [c for c in val.columns if c.startswith("emb_")]
        val_graph_features = base_graph_cols + emb_cols
        X_val_graph = val[val_graph_features].fillna(0)
        if graph_model is not None:
            graph_scaler, graph_clf = graph_model
            X_val_graph_scaled = graph_scaler.transform(X_val_graph)
            val["graph_score"] = graph_clf.predict_proba(X_val_graph_scaled)[:, 1]
        else:
            val["graph_score"] = 0.0

        # Step 9b: Pairwise ranking model (numeric features only)
        pairwise_features = [
            "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp",
            "shock_index", "pulse_pressure", "complexity_score",
            "congestion_score", "active_patients", "wait_time_proxy",
            "arrival_hour", "arrival_day",
        ]
        scaler, pairwise_model = train_pairwise_ranker(train, val, pairwise_features, "arrival_hour_bucket")
        val["pairwise_score"] = score_pairwise(val, pairwise_features, scaler, pairwise_model)
        test["pairwise_score"] = score_pairwise(test, pairwise_features, scaler, pairwise_model)
    else:
        graph_params = {}
        graph_cv = {"params": [], "mean_test_score": []}
        val["graph_score"] = 0.0
        test["graph_score"] = 0.0
        val["pairwise_score"] = 0.0
        test["pairwise_score"] = 0.0

    # Step 10: Evaluate ranking metrics
    metrics = {
        "config": {
            "label_strategy": cfg.label_strategy,
            "threshold_metric": cfg.threshold_metric,
            "threshold_range": [cfg.threshold_min, cfg.threshold_max, cfg.threshold_step],
        },
        "baseline": {},
        "context": {"best_params": context_params},
        "matrix_factorization": {"best_params": {"n_factors": mf_model["n_factors"]}},
        "graph": {"best_params": graph_params},
        "pairwise": {},
    }
    group_col = "arrival_hour_bucket"
    for k in cfg.k_values:
        metrics["baseline"][f"k={k}"] = evaluate_at_k(test, "rule_score", "is_critical", group_col, k)
        metrics["baseline"][f"ndcg@{k}"] = evaluate_ndcg_at_k(test, "rule_score", "is_critical", group_col, k)
        metrics["context"][f"k={k}"] = evaluate_at_k(test, "context_score", "is_critical", group_col, k)
        metrics["context"][f"ndcg@{k}"] = evaluate_ndcg_at_k(test, "context_score", "is_critical", group_col, k)
        metrics["matrix_factorization"][f"k={k}"] = evaluate_at_k(test, "mf_score", "is_critical", group_col, k)
        metrics["matrix_factorization"][f"ndcg@{k}"] = evaluate_ndcg_at_k(test, "mf_score", "is_critical", group_col, k)
        metrics["graph"][f"k={k}"] = evaluate_at_k(test, "graph_score", "is_critical", group_col, k)
        metrics["graph"][f"ndcg@{k}"] = evaluate_ndcg_at_k(test, "graph_score", "is_critical", group_col, k)
        metrics["pairwise"][f"k={k}"] = evaluate_at_k(test, "pairwise_score", "is_critical", group_col, k)
        metrics["pairwise"][f"ndcg@{k}"] = evaluate_ndcg_at_k(test, "pairwise_score", "is_critical", group_col, k)

    # Step 10b: Threshold tuning for critical-prioritized classification
    val_true = val["is_critical"].values
    ctx_t, ctx_val_score = find_best_threshold(
        val["context_score"].values,
        val_true,
        cfg.threshold_metric,
        cfg.threshold_min,
        cfg.threshold_max,
        cfg.threshold_step,
    )
    g_t, g_val_score = find_best_threshold(
        val["graph_score"].values,
        val_true,
        cfg.threshold_metric,
        cfg.threshold_min,
        cfg.threshold_max,
        cfg.threshold_step,
    )
    p_t, p_val_score = find_best_threshold(
        val["pairwise_score"].values,
        val_true,
        cfg.threshold_metric,
        cfg.threshold_min,
        cfg.threshold_max,
        cfg.threshold_step,
    )
    metrics["context"]["best_threshold"] = ctx_t
    metrics["context"]["best_threshold_score"] = ctx_val_score
    mf_t, mf_val_score = find_best_threshold(
        val["mf_score"].values,
        val_true,
        cfg.threshold_metric,
        cfg.threshold_min,
        cfg.threshold_max,
        cfg.threshold_step,
    )
    metrics["matrix_factorization"]["best_threshold"] = mf_t
    metrics["matrix_factorization"]["best_threshold_score"] = mf_val_score
    metrics["graph"]["best_threshold"] = g_t
    metrics["graph"]["best_threshold_score"] = g_val_score
    metrics["pairwise"]["best_threshold"] = p_t
    metrics["pairwise"]["best_threshold_score"] = p_val_score

    # Step 10c: Classification metrics for model selection
    y_true = test["is_critical"].values
    metrics["baseline"]["roc_auc"] = float(roc_auc_score(y_true, test["rule_score"]))
    metrics["baseline"]["pr_auc"] = float(average_precision_score(y_true, test["rule_score"]))
    metrics["baseline"]["f1"] = float(f1_score(y_true, (test["rule_score"] > 0).astype(int)))
    metrics["baseline"]["accuracy"] = float(
        accuracy_score(y_true, (test["rule_score"] > 0).astype(int))
    )

    metrics["context"]["roc_auc"] = float(roc_auc_score(y_true, test["context_score"]))
    metrics["context"]["pr_auc"] = float(average_precision_score(y_true, test["context_score"]))
    metrics["context"]["f1"] = float(f1_score(y_true, (test["context_score"] > ctx_t).astype(int)))
    metrics["context"]["f2"] = float(
        fbeta_score_safe(y_true, (test["context_score"] > ctx_t).astype(int), beta=2.0)
    )
    metrics["context"]["accuracy"] = float(
        accuracy_score(y_true, (test["context_score"] > ctx_t).astype(int))
    )

    metrics["matrix_factorization"]["roc_auc"] = float(roc_auc_score(y_true, test["mf_score"]))
    metrics["matrix_factorization"]["pr_auc"] = float(average_precision_score(y_true, test["mf_score"]))
    metrics["matrix_factorization"]["f1"] = float(f1_score(y_true, (test["mf_score"] > mf_t).astype(int)))
    metrics["matrix_factorization"]["f2"] = float(
        fbeta_score_safe(y_true, (test["mf_score"] > mf_t).astype(int), beta=2.0)
    )
    metrics["matrix_factorization"]["accuracy"] = float(
        accuracy_score(y_true, (test["mf_score"] > mf_t).astype(int))
    )

    metrics["graph"]["roc_auc"] = float(roc_auc_score(y_true, test["graph_score"]))
    metrics["graph"]["pr_auc"] = float(average_precision_score(y_true, test["graph_score"]))
    metrics["graph"]["f1"] = float(f1_score(y_true, (test["graph_score"] > g_t).astype(int)))
    metrics["graph"]["f2"] = float(
        fbeta_score_safe(y_true, (test["graph_score"] > g_t).astype(int), beta=2.0)
    )
    metrics["graph"]["accuracy"] = float(
        accuracy_score(y_true, (test["graph_score"] > g_t).astype(int))
    )

    metrics["pairwise"]["roc_auc"] = float(roc_auc_score(y_true, test["pairwise_score"]))
    metrics["pairwise"]["pr_auc"] = float(average_precision_score(y_true, test["pairwise_score"]))
    metrics["pairwise"]["f1"] = float(f1_score(y_true, (test["pairwise_score"] > p_t).astype(int)))
    metrics["pairwise"]["f2"] = float(
        fbeta_score_safe(y_true, (test["pairwise_score"] > p_t).astype(int), beta=2.0)
    )
    metrics["pairwise"]["accuracy"] = float(
        accuracy_score(y_true, (test["pairwise_score"] > p_t).astype(int))
    )

    # Step 11: Save metrics
    os.makedirs(cfg.output_dir, exist_ok=True)
    output_path = f"{cfg.output_dir}/triage_metrics.json"
    plot_model_selection(
        context_cv,
        "Context Model Selection (ROC-AUC vs C)",
        f"{cfg.output_dir}/model_selection_context.png",
    )
    plot_model_selection(
        graph_cv,
        "Graph Model Selection (ROC-AUC vs C)",
        f"{cfg.output_dir}/model_selection_graph.png",
    )
    with open(output_path, "w") as f:
        json.dump(metrics, f, indent=2)

    # Step 12: Print summary
    print("=== Classification Report (Context Model) ===")
    print(
        classification_report(
            test["is_critical"],
            (test["context_score"] > ctx_t).astype(int),
        )
    )
    print("=== Classification Report (Graph Model) ===")
    print(
        classification_report(
            test["is_critical"],
            (test["graph_score"] > g_t).astype(int),
        )
    )
    print(f"\nMetrics saved to: {output_path}")


if __name__ == "__main__":
    main()


## 2. Embedded Figures and Visual Analysis Outputs
All figures below are embedded in this notebook for standalone submission.


### `reports/model_selection_context.png`


In [ ]:
# Embedded figure: reports/model_selection_context.png


### `reports/model_selection_graph.png`


In [ ]:
# Embedded figure: reports/model_selection_graph.png


### `reports/eda/count_acuity.png`


In [ ]:
# Embedded figure: reports/eda/count_acuity.png


### `reports/eda/count_arrival_hour.png`


In [ ]:
# Embedded figure: reports/eda/count_arrival_hour.png


### `reports/eda/count_arrival_day.png`


In [ ]:
# Embedded figure: reports/eda/count_arrival_day.png


### `reports/eda/count_disposition.png`


In [ ]:
# Embedded figure: reports/eda/count_disposition.png


### `reports/eda/count_chiefcomplaint.png`


In [ ]:
# Embedded figure: reports/eda/count_chiefcomplaint.png


### `reports/eda/count_transport.png`


In [ ]:
# Embedded figure: reports/eda/count_transport.png


### `reports/eda/dist_temperature.png`


In [ ]:
# Embedded figure: reports/eda/dist_temperature.png


### `reports/eda/dist_heartrate.png`


In [ ]:
# Embedded figure: reports/eda/dist_heartrate.png


### `reports/eda/dist_resprate.png`


In [ ]:
# Embedded figure: reports/eda/dist_resprate.png


### `reports/eda/dist_o2sat.png`


In [ ]:
# Embedded figure: reports/eda/dist_o2sat.png


### `reports/eda/dist_sbp.png`


In [ ]:
# Embedded figure: reports/eda/dist_sbp.png


### `reports/eda/dist_dbp.png`


In [ ]:
# Embedded figure: reports/eda/dist_dbp.png


### `reports/eda/missingness_top20.png`


In [ ]:
# Embedded figure: reports/eda/missingness_top20.png


### `reports/eda/corr_heatmap.png`


In [ ]:
# Embedded figure: reports/eda/corr_heatmap.png


### `reports/eda/boxplot_vitals.png`


In [ ]:
# Embedded figure: reports/eda/boxplot_vitals.png


### `reports/eda/violin_heartrate_by_acuity.png`


In [ ]:
# Embedded figure: reports/eda/violin_heartrate_by_acuity.png


### `reports/eda/violin_resprate_by_acuity.png`


In [ ]:
# Embedded figure: reports/eda/violin_resprate_by_acuity.png


### `reports/eda/violin_o2sat_by_acuity.png`


In [ ]:
# Embedded figure: reports/eda/violin_o2sat_by_acuity.png


### `reports/eda/violin_sbp_by_acuity.png`


In [ ]:
# Embedded figure: reports/eda/violin_sbp_by_acuity.png


### `reports/eda/pairplot_vitals.png`


In [ ]:
# Embedded figure: reports/eda/pairplot_vitals.png


### `reports/visuals/pipeline_overview.png`


In [ ]:
# Embedded figure: reports/visuals/pipeline_overview.png


### `reports/visuals/auc_comparison.png`


In [ ]:
# Embedded figure: reports/visuals/auc_comparison.png


### `reports/visuals/fscore_comparison.png`


In [ ]:
# Embedded figure: reports/visuals/fscore_comparison.png


### `reports/visuals/accuracy_comparison.png`


In [ ]:
# Embedded figure: reports/visuals/accuracy_comparison.png


### `reports/visuals/ranking_comparison.png`


In [ ]:
# Embedded figure: reports/visuals/ranking_comparison.png


### `reports/visuals/threshold_summary.png`


In [ ]:
# Embedded figure: reports/visuals/threshold_summary.png


### `reports/visuals/class_balance_pie.png`


In [ ]:
# Embedded figure: reports/visuals/class_balance_pie.png


### `reports/visuals/split_sizes_bar.png`


In [ ]:
# Embedded figure: reports/visuals/split_sizes_bar.png


### `reports/visuals/top_disposition_bar.png`


In [ ]:
# Embedded figure: reports/visuals/top_disposition_bar.png


### `reports/visuals/top_complaints_bar.png`


In [ ]:
# Embedded figure: reports/visuals/top_complaints_bar.png


### `reports/visuals/categorical_cardinality.png`


In [ ]:
# Embedded figure: reports/visuals/categorical_cardinality.png


### `reports/visuals/model_profile_lines.png`


In [ ]:
# Embedded figure: reports/visuals/model_profile_lines.png


## 3. Reproducibility Commands
If runtime/data are available, the full pipeline can be rerun with:

```bash
python src/data_preprocessing.py
python src/feature_engineering.py
python src/eda_visualization.py
python src/triage_pipeline.py
```
